## Config

### Install

In [0]:
%run ./00_utility

In [0]:
import re
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

### Parameters

In [0]:
catalog = get_catalog()
print("Catalog: ", catalog)
# TABLES
PREDICTIONS_TABLE = f"{catalog}.whatif.out_palinsesto_predict"
HIST_TABLE = f"{catalog}.whatif.storico_programmi"
DELTA_TABLE = f"{catalog}.whatif.output_palinsesto_delta"
VW_DELTA_MONITOR = f"{catalog}.whatif.vw_output_palinsesto_delta_monitor"
# Parametri per le query nelle celle SQL (:param)
dbutils.widgets.text("DELTA_TABLE", DELTA_TABLE)
dbutils.widgets.text("VW_DELTA_MONITOR", VW_DELTA_MONITOR)

# PARAMETERS
CHANNELS = ['Rai 1', 'Rai 2', 'Rai 3']
AUDITEL_LAG_DAYS = 3 # 2 # giorni di lag per i dati di share Auditel
TOLERANCE_MATCH_START_TIME = 18000 # 30 minuti
START_PRIMETIME = 20 * 3600 + 30 * 60 # 20.30
TOLERANCE_START_PRIME_TIME = 18000 # 30 minuti
HISTORICAL_PROGRAMMING_CUTOFF_DATE = (pd.Timestamp.today() - pd.DateOffset(years=1)).strftime("%Y-%m-%d") # data da cui partire per estrarre lo storico auditel

## Loading Palinsesto Storico e Futuro

In [0]:
# Load future programming schedule with share predictions
output = spark.table(PREDICTIONS_TABLE).select(
    'Data',
    'Programma',
    'Canale',
    'ORA_INIZIO_TRX',
    'ORA_FINE_TRX',
    'orario_inizio',
    'orario_fine',
    'share_predetto',
    'programma_norm',
    'share_manuale'
)

# Filtro per avere solo i canali Rai e i dati per i quali riusciremo ad incrociare lo share Auditel
output = (
    output.filter(F.col('Canale').isin(CHANNELS))
          .filter(F.col('Data') == F.current_date() - F.expr(f'INTERVAL {AUDITEL_LAG_DAYS} DAY'))
          .select(
              'Data',
              'Programma',
              'Canale',
              'ORA_INIZIO_TRX',
              'ORA_FINE_TRX',
              'orario_inizio',
              'orario_fine',
              'share_predetto',
              'programma_norm',
              'share_manuale'
          )
)

In [0]:
display(output.orderBy(F.col("Data").asc(), F.col("orario_inizio").asc()))

In [0]:
# Load historical programming 
df_programmi = spark.table(HIST_TABLE)

# Apply filters to select: 
# - only rai TV channels 
# - solo l'ultimo giorno ricevuto con i dati di share Auditel
# - solo programmi in Prime Time (con un po' di tolleranza visti gli orari al minuto, per permettere alla logica di fallback di intercettare anche i programmi vicini)
df_programmi = (
    df_programmi
    .filter(F.col('Canale').isin(CHANNELS))
    .filter(F.col('Data') == F.current_date() - F.expr(f'INTERVAL {AUDITEL_LAG_DAYS} DAY'))
    .filter(F.col('ORA_INIZIO_TRX') >= START_PRIMETIME - TOLERANCE_START_PRIME_TIME)
    .select('Data', 'programma_norm', 'Canale', 'ORA_INIZIO_TRX', 'Share')
)

# Convert to Pandas and back to Spark DataFrame
df_programmi = df_programmi.toPandas()
df_programmi = spark.createDataFrame(df_programmi)
display(df_programmi.orderBy(F.col("ORA_INIZIO_TRX").desc()))

## Calcolo share fallback e share delta

### Match righe predict con righe Auditel

In [0]:
# Aggiungiamo una colonna di supporto per tenere traccia dell'ora originale di inizio del programma prima dell'applicazione della tolleranza
df_programmi_with_hist = df_programmi.withColumn(
    "hist_ORA_INIZIO_TRX", F.col("ORA_INIZIO_TRX")
)

# Left join future programming with historical programming so that for each program we have both predicted and actual share
df_delta = output.join(
    df_programmi_with_hist,
    on=[
        output["Canale"] == df_programmi_with_hist["Canale"],
        output["Data"] == df_programmi_with_hist["Data"],
        output["programma_norm"] == df_programmi_with_hist["programma_norm"],
        output["ORA_INIZIO_TRX"].between(
            df_programmi_with_hist["ORA_INIZIO_TRX"] - TOLERANCE_MATCH_START_TIME,
            df_programmi_with_hist["ORA_INIZIO_TRX"] + TOLERANCE_MATCH_START_TIME,
        ),
    ],
    how="left",
).drop(
    *[df_programmi_with_hist[k] for k in ["Data", "Canale", "ORA_INIZIO_TRX", "programma_norm"]]
)

# Applicando la tolleranza potrebbero esserci dei duplicati - in questo caso teniamo il record di auditel con l'orario di inizio più vicino (anche se non esattamente uguale) a quello di tivu tivu
df_delta = df_delta.withColumn(
    "time_diff", F.abs(F.col("ORA_INIZIO_TRX") - F.col("hist_ORA_INIZIO_TRX"))
)
w = Window.partitionBy("Canale", "Data", "programma_norm", "ORA_INIZIO_TRX").orderBy("time_diff")
df_delta = (
    df_delta.withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn", "time_diff", "hist_ORA_INIZIO_TRX")
)

# Selezionamo solo alcune colonne
df_delta = (
    df_delta.select(
        "Canale",
        "Data",
        "Programma",
        "ORA_INIZIO_TRX",
        "ORA_FINE_TRX",
        "orario_inizio",
        "orario_fine",
        "share_predetto",
        "share_manuale",
        "Share"
    )
    .withColumnRenamed("Share", "share_reale")
)

In [0]:
display(df_delta.orderBy(F.col("Data").asc(), F.col("orario_inizio").asc()))

### Calcolo share fallback

In [0]:
# Fallback fill per share_reale null su date con lo share Auditel disponibile.
# Il match per nome non ha trovato il programma nello storico, ma il programma esiste comunque - ha solo un nome leggermente diverso tra le due sorgenti.
# Proviamo a recuperare lo share abbinando per (Canale, Data, ORA_INIZIO_TRX) con tolleranza.

# --- Preparazione lookup da storico ---
df_hist_for_fill = (
    df_programmi_with_hist
    .select(
        F.col("Canale").alias("hist_Canale"),
        F.col("Data").alias("hist_Data"),
        F.col("ORA_INIZIO_TRX").alias("hist_ORA_INIZIO_TRX"),
        F.col("Share").alias("share_fallback"),
        F.col("programma_norm").alias("programma_fallback")
    )
)

# --- Suddivisione di df_delta ---
# Righe da recuperare: share_reale null
condition_fill = (
    F.col("share_reale").isNull()
)
df_delta_to_fill = df_delta.filter(condition_fill)
# Righe da non recuperare: match avvenuto con successo usando anche il nome del programma
df_delta_ok = (
    df_delta.filter(~condition_fill)
    .withColumn("share_fallback", F.lit(None).cast("double"))
    .withColumn("programma_fallback", F.lit(None).cast("string"))
    .drop("hist_Canale", "hist_Data", "hist_ORA_INIZIO_TRX", "ORA_INIZIO_TRX", "ORA_FINE_TRX")
)

# --- Join per time slot con tolleranza ---
df_delta_filled = (
    df_delta_to_fill
    .join(
        df_hist_for_fill,
        on=[
            F.col("Canale") == F.col("hist_Canale"),
            F.col("Data")   == F.col("hist_Data"),
            F.col("ORA_INIZIO_TRX").between(
                F.col("hist_ORA_INIZIO_TRX") - TOLERANCE_MATCH_START_TIME,
                F.col("hist_ORA_INIZIO_TRX") + TOLERANCE_MATCH_START_TIME,
            ),
        ],
        how="left"
    )
    .withColumn("time_diff_fb", F.abs(F.col("ORA_INIZIO_TRX") - F.col("hist_ORA_INIZIO_TRX")))
)

w_top5 = Window.partitionBy("Canale", "Data", "Programma", "ORA_INIZIO_TRX").orderBy(F.col("time_diff_fb").asc())
df_top5_per_programma = (
    df_delta_filled
    .withColumn("rn_top5", F.row_number().over(w_top5))
    .filter(F.col("rn_top5") <= 5)
    .select(
        "Canale", "Data", "Programma", "ORA_INIZIO_TRX", "ORA_FINE_TRX", "orario_inizio", "orario_fine", "share_fallback", "programma_fallback", "hist_ORA_INIZIO_TRX", "time_diff_fb"
    )
)
display(df_top5_per_programma.orderBy(F.col("Data").asc(), F.col("Programma").asc(), F.col("time_diff_fb").asc()))

In [0]:
# Deduplica: se più storici rientrano nella tolleranza, tieni quello con ORA_INIZIO_TRX più vicino
w_fb = Window.partitionBy("Canale", "Data", "ORA_INIZIO_TRX").orderBy("time_diff_fb")
df_delta_filled = (
    df_delta_filled
    .withColumn("rn_fb", F.row_number().over(w_fb))
    .filter(F.col("rn_fb") == 1)
    .drop("rn_fb", "time_diff_fb", "hist_Canale", "hist_Data", "hist_ORA_INIZIO_TRX", "ORA_INIZIO_TRX", "ORA_FINE_TRX")
)

# --- Ricostruzione df_delta ---
df_delta = df_delta_filled.unionByName(df_delta_ok)

In [0]:
display(df_delta.filter(F.col("share_reale").isNull()).orderBy(F.col("Data").asc(), F.col("orario_inizio").asc()))

### Calcolo delta share

In [0]:
# Ricostruzione df_delta pulito per l'output:
# - se share_reale e' null ma share_fallback e' valorizzato, usa il fallback come share_reale
# - altrimenti mantieni share_reale com'e'
# Rimuove le colonne di lavoro e ricalcola delta_share
df_delta = (
    df_delta
    .withColumn("share_reale", F.coalesce(F.col("share_reale"), F.col("share_fallback")))
    .drop("share_fallback", "programma_fallback", "share_reale_fallback")
    .withColumn(
        "delta_share",
        F.round(
            F.col("share_reale") - F.coalesce(F.col("share_manuale"), F.col("share_predetto")),
            4,
        ),
    )
)

In [0]:
display(df_delta.orderBy(F.col("Data").asc(), F.col("orario_inizio").asc()))

## Output

### Tabella Delta

In [0]:
df_delta = df_delta.withColumn(
    'ID',
    F.concat(
        F.col('Canale'),
        F.lit('_'),
        F.col('Data'),
        F.lit('_'),
        F.col('Programma'),
        F.lit('_'),
        F.col('orario_inizio')
    )
)

# Salviamo le date di messa in onda che stiamo elaborando (per i filtri sulle tabelle del catalog in fase di salvataggio)
date_in_delta = [row.Data for row in df_delta.select("Data").distinct().collect()]
date_list = ",".join([f"'{d}'" for d in date_in_delta])

# if table doesn't exist in UC --> create it
if not spark.catalog.tableExists(DELTA_TABLE):
    df_delta.write.format("delta").saveAsTable(DELTA_TABLE)
else:
    # altrimenti replaceWhere sulla tabella in base alla data
    (
        df_delta.write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"Data IN ({date_list})")
        .saveAsTable(DELTA_TABLE)
    )

display(spark.table(DELTA_TABLE).filter(F.col("Data").isin(date_in_delta)).orderBy("Data", "orario_inizio"))

### Vista Monitoraggio Delta

In [0]:
# Tabella delta escludendo i programmi piu' recenti (servono 3 giorni per ricevere lo share_reale) e piu' vecchi del 10 giugno 2026
df_monitor = (
    spark.table(DELTA_TABLE)
    .filter((F.col("Data") >= F.add_months(F.current_date(), -1)) & (F.col("Data") <= F.date_sub(F.current_date(), AUDITEL_LAG_DAYS)))
    .withColumn("delta_share", F.col("share_reale") - F.col("share_predetto"))
)

# Tabella con i label per dividere la qualita' delle predizioni + tracciamento null
df_accuracy = (
    df_monitor.groupBy("Data")
    .agg(
        F.count("*").alias("Totale_Programmi"),
        F.sum(F.when((F.col("share_predetto").isNotNull()) & (F.col("share_reale").isNotNull()), 1).otherwise(0)).alias("Totale_Predizioni"),
        F.sum(F.when(F.abs(F.col("delta_share")) <= 0.02, 1).otherwise(0)).alias("Sotto_2"),
        F.sum(F.when((F.abs(F.col("delta_share")) > 0.02) & (F.abs(F.col("delta_share")) <= 0.05), 1).otherwise(0)).alias("Tra_2_e_5"),
        F.sum(F.when((F.abs(F.col("delta_share")) > 0.05) & (F.abs(F.col("delta_share")) <= 0.10), 1).otherwise(0)).alias("Tra_5_e_10"),
        F.sum(F.when(F.abs(F.col("delta_share")) > 0.10, 1).otherwise(0)).alias("Oltre_10"),
        F.sum(F.when(F.col("share_predetto").isNull(), 1).otherwise(0)).alias("Null_Predetto"),
        F.collect_list(F.when(F.abs(F.col("delta_share")) <= 0.02, F.col("ID"))).alias("IDs_Sotto_2"),
        F.collect_list(F.when((F.abs(F.col("delta_share")) > 0.02) & (F.abs(F.col("delta_share")) <= 0.05), F.col("ID"))).alias("IDs_Tra_2_e_5"),
        F.collect_list(F.when((F.abs(F.col("delta_share")) > 0.05) & (F.abs(F.col("delta_share")) <= 0.10), F.col("ID"))).alias("IDs_Tra_5_e_10"),
        F.collect_list(F.when(F.abs(F.col("delta_share")) > 0.10, F.col("ID"))).alias("IDs_Oltre_10"),
    )
    .withColumns({
        "Pct_Sotto_2": F.round(F.col("Sotto_2") / F.col("Totale_Predizioni") * 100, 1),
        "Pct_Tra_2_e_5": F.round(F.col("Tra_2_e_5") / F.col("Totale_Predizioni") * 100, 1),
        "Pct_Tra_5_e_10": F.round(F.col("Tra_5_e_10") / F.col("Totale_Predizioni") * 100, 1),
        "Pct_Oltre_10": F.round(F.col("Oltre_10") / F.col("Totale_Predizioni") * 100, 1),
        "Pct_Null": F.round(F.col("Null_Predetto") / F.col("Totale_Programmi") * 100, 1),
    })
    .orderBy("Data")
)

display(df_accuracy.orderBy(F.col("Data").desc()))

# Grafico 1: accuracy cumulativa
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
import pandas as pd

pdf_acc = df_accuracy.toPandas().sort_values("Data")

pct_cols = ["Pct_Sotto_2", "Pct_Tra_2_e_5", "Pct_Tra_5_e_10", "Pct_Oltre_10"]
for col in pct_cols:
    pdf_acc[f"CUM_{col}"] = pdf_acc[col].expanding(min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 6))
colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
labels = ["\u22642%", "2-5%", "5-10%", ">10%"]

for col, color, label in zip(pct_cols, colors, labels):
    ax.plot(pdf_acc["Data"], pdf_acc[f"CUM_{col}"], color=color, label=label, linewidth=2)
    ax.scatter(pdf_acc["Data"], pdf_acc[col], color=color, alpha=0.3, s=20)

# ax.set_xlabel("Data")
ax.set_ylabel("% Predizioni")
ax.set_title("Accuratezza modello previsioni Prime Time - media cumulativa")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Grafico 2: numero predizioni effettuate per giorno
fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(pdf_acc["Data"], pdf_acc["Totale_Predizioni"], color="#3498db", alpha=0.7)
ax.bar_label(bars, fmt="%d", fontsize=8, padding=2)
ax.plot(pdf_acc["Data"], pdf_acc["Totale_Predizioni"].expanding(min_periods=1).mean(), color="#2c3e50", linestyle="--", linewidth=1.5, label="Media cumulativa")

ax.set_ylabel("Conteggio")
ax.set_title("Numero predizioni Prime Time effettuate per giorno")
ax.legend(loc="upper right")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Grafico 3: null/totale programmi nel tempo
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(pdf_acc["Data"], pdf_acc["Pct_Null"], color="#e74c3c", linewidth=2)
ax.scatter(pdf_acc["Data"], pdf_acc["Pct_Null"], color="#e74c3c", alpha=0.4, s=20)

# ax.set_xlabel("Data")
ax.set_ylabel("% Null")
ax.set_title("% Predizioni null sul totale programmi Prime Time del giorno")
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Grafico 4: conteggio null nel tempo

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(pdf_acc["Data"], pdf_acc["Null_Predetto"], color="#95a5a6", alpha=0.7)
ax.bar_label(bars, fmt="%d", fontsize=8, padding=2)
ax.plot(pdf_acc["Data"], pdf_acc["Null_Predetto"].expanding(min_periods=1).mean(), color="#2c3e50", linestyle="--", linewidth=1.5, label="Media cumulativa")

# ax.set_xlabel("Data")
ax.set_ylabel("Conteggio")
ax.set_title("Numero predizioni Prime Time null per giorno")
ax.legend(loc="upper right")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
# Usiamo spark.sql perche' DDL non supporta variabili
spark.sql(f"""
-- CREATE OR REPLACE VIEW {VW_DELTA_MONITOR}
CREATE VIEW IF NOT EXISTS {VW_DELTA_MONITOR}
(
    Data COMMENT 'Data di messa in onda',
    Totale_Programmi COMMENT 'Numero totale di programmi per quel giorno (con dati Auditel)',
    Totale_Predizioni COMMENT 'Numero di programmi con share_predetto e share_reale non nulli (predizioni valutabili)',
    Sotto_2 COMMENT 'Predizioni con abs(share_predetto - share_reale) <= 0.02',
    Tra_2_e_5 COMMENT 'Predizioni con 0.02 < abs(share_predetto - share_reale) <= 0.05',
    Tra_5_e_10 COMMENT 'Predizioni con 0.05 < abs(share_predetto - share_reale) <= 0.10',
    Oltre_10 COMMENT 'Predizioni con abs(share_predetto - share_reale) > 0.10',
    Null_Predetto COMMENT 'Programmi con share_predetto NULL',
    Pct_Sotto_2 COMMENT 'Percentuale sul totale predizioni con errore <= 0.02',
    Pct_Tra_2_e_5 COMMENT 'Percentuale sul totale predizioni con errore tra 0.02 e 0.05',
    Pct_Tra_5_e_10 COMMENT 'Percentuale sul totale predizioni con errore tra 0.05 e 0.10',
    Pct_Oltre_10 COMMENT 'Percentuale sul totale predizioni con errore > 0.10',
    Pct_Null COMMENT 'Percentuale di null sul totale programmi del giorno',
    IDs_Sotto_2 COMMENT 'IDs delle predizioni nella banda <= 2%',
    IDs_Tra_2_e_5 COMMENT 'IDs delle predizioni nella banda 2-5%',
    IDs_Tra_5_e_10 COMMENT 'IDs delle predizioni nella banda 5-10%',
    IDs_Oltre_10 COMMENT 'IDs delle predizioni nella banda > 10%'
)
COMMENT 'Vista per il monitoraggio accuratezza modello predittivo - confronto share_predetto vs share_reale'
AS
WITH base AS (
    SELECT
        ID,
        Data,
        share_predetto,
        share_reale,
        ABS(share_predetto - share_reale) AS abs_error
    FROM {DELTA_TABLE}
    WHERE 1=1
      AND Data >= '2026-06-15'
      AND Data <= DATEADD(DAY, -3, CURRENT_DATE())
)
SELECT
    Data,
    COUNT(*) AS Totale_Programmi,
    SUM(CASE WHEN share_predetto IS NOT NULL AND share_reale IS NOT NULL THEN 1 ELSE 0 END) AS Totale_Predizioni,
    SUM(CASE WHEN abs_error <= 0.02 THEN 1 ELSE 0 END) AS Sotto_2,
    SUM(CASE WHEN abs_error > 0.02 AND abs_error <= 0.05 THEN 1 ELSE 0 END) AS Tra_2_e_5,
    SUM(CASE WHEN abs_error > 0.05 AND abs_error <= 0.10 THEN 1 ELSE 0 END) AS Tra_5_e_10,
    SUM(CASE WHEN abs_error > 0.10 THEN 1 ELSE 0 END) AS Oltre_10,
    SUM(CASE WHEN share_predetto IS NULL THEN 1 ELSE 0 END) AS Null_Predetto,
    ROUND(SUM(CASE WHEN abs_error <= 0.02 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN share_predetto IS NOT NULL AND share_reale IS NOT NULL THEN 1 ELSE 0 END), 0) * 100, 1) AS Pct_Sotto_2,
    ROUND(SUM(CASE WHEN abs_error > 0.02 AND abs_error <= 0.05 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN share_predetto IS NOT NULL AND share_reale IS NOT NULL THEN 1 ELSE 0 END), 0) * 100, 1) AS Pct_Tra_2_e_5,
    ROUND(SUM(CASE WHEN abs_error > 0.05 AND abs_error <= 0.10 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN share_predetto IS NOT NULL AND share_reale IS NOT NULL THEN 1 ELSE 0 END), 0) * 100, 1) AS Pct_Tra_5_e_10,
    ROUND(SUM(CASE WHEN abs_error > 0.10 THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN share_predetto IS NOT NULL AND share_reale IS NOT NULL THEN 1 ELSE 0 END), 0) * 100, 1) AS Pct_Oltre_10,
    ROUND(SUM(CASE WHEN share_predetto IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS Pct_Null,
    COLLECT_LIST(CASE WHEN abs_error <= 0.02 THEN ID END) AS IDs_Sotto_2,
    COLLECT_LIST(CASE WHEN abs_error > 0.02 AND abs_error <= 0.05 THEN ID END) AS IDs_Tra_2_e_5,
    COLLECT_LIST(CASE WHEN abs_error > 0.05 AND abs_error <= 0.10 THEN ID END) AS IDs_Tra_5_e_10,
    COLLECT_LIST(CASE WHEN abs_error > 0.10 THEN ID END) AS IDs_Oltre_10
FROM base
GROUP BY Data
ORDER BY Data DESC;
"""
)

In [0]:
%sql
SELECT * FROM IDENTIFIER(:VW_DELTA_MONITOR) 